# LinkedIn post classifier

Train a binary classifier: **useful = 1**, **useless = 0**.

In [1]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences


test_usefull = "../data/test/usefull_posts.cvs"
test_useless = "../data/test/useless_posts.cvs"
training_usefull = "../data/training/usefull_posts.cvs"
training_useless = "../data/training/useless_posts.cvs"

VectorSize = 300


## 1. Load and clean

Each post is one example, regardless of word count. Lowercasing and removing punctuation make the vocabulary simpler. The trade-off is that names like `C++` lose their symbols.


In [2]:
useful_train = pd.read_csv(training_usefull)['post_content'].dropna().astype(str).tolist()
useless_train = pd.read_csv(training_useless)['post_content'].dropna().astype(str).tolist()
useful_test = pd.read_csv(test_usefull)['post_content'].dropna().astype(str).tolist()
useless_test = pd.read_csv(test_useless)['post_content'].dropna().astype(str).tolist()

def clean(text):
    return ' '.join(re.findall(r'\b\w+\b', text.lower()))

texts = [clean(text) for text in useful_test + useful_train + useless_train + useless_test]
labels = np.array([1] * (len(useful_test) + len(useful_train)) + [0] * (len(useless_test) + len(useless_train)))
print(f'{len(useful_test) + len(useful_train)} useful, {len(useless_test) + len(useless_train)} useless posts')


489 useful, 440 useless posts


## 2. Split and encode

Reserve 20% for validation. `stratify` keeps the class proportions similar. Fit the tokenizer on training text only so validation vocabulary does not leak into training. Padding gives every post the same input length; longer posts are cut off.


In [3]:
train_texts, val_texts, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
MAX_LEN = 350  # Room for most approximately 300-word posts.
tokenizer = Tokenizer(oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(train_texts)

def encode(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

x_train, x_val = encode(train_texts), encode(val_texts)


## 3. Train Word2Vec

Word2Vec learns a 100-number vector for each word from its neighbors in the training posts. `min_count=1` keeps rare technical words, but also typos. The embedding matrix aligns vectors with tokenizer IDs; row 0 is padding.


In [4]:
w2v = Word2Vec([text.split() for text in train_texts], vector_size=VectorSize, min_count=0, epochs=10)
embeddings = np.zeros((len(tokenizer.word_index) + 1, 300), dtype=np.float32)
for word, index in tokenizer.word_index.items():
    if word in w2v.wv:
        embeddings[index] = w2v.wv[word]


## 4. Train the CNN

`Conv1D` looks for patterns across five neighboring words. Pooling retains strong pattern signals. The sigmoid output is the estimated probability that a post is useful. Training the embeddings lets them adapt to this task, but may overfit; early stopping restores the weights from the best validation-loss epoch.


In [5]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(len(embeddings), VectorSize, weights=[embeddings], trainable=True),
    Conv1D(64, 20, activation='relu'),
    MaxPooling1D(2),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(
    x_train, y_train, validation_data=(x_val, y_val),
    epochs=20, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step - accuracy: 0.5478 - loss: 0.7293 - val_accuracy: 0.5376 - val_loss: 0.7384
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 136ms/step - accuracy: 0.6568 - loss: 0.5939 - val_accuracy: 0.6559 - val_loss: 0.5907
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 135ms/step - accuracy: 0.7577 - loss: 0.4976 - val_accuracy: 0.7097 - val_loss: 0.5891
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.7900 - loss: 0.4447 - val_accuracy: 0.6613 - val_loss: 0.6079
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 143ms/step - accuracy: 0.8910 - loss: 0.3404 - val_accuracy: 0.6989 - val_loss: 0.6094


## 5. Evaluate and use

Accuracy is the overall fraction correct. Precision and recall show how the model performs for each class. A probability of 0.5 or higher is classified as useful.


In [6]:
useful_test = pd.read_csv(test_usefull)["post_content"].dropna().astype(str).tolist()
useless_test = pd.read_csv(test_useless)["post_content"].dropna().astype(str).tolist()

test_texts = [clean(text) for text in useful_test + useless_test]
x_test = encode(test_texts)  # Uses the tokenizer fitted on training data.
y_test = np.array([1] * len(useful_test) + [0] * len(useless_test))

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
predictions = (model.predict(x_test, verbose=0).ravel() >= 0.5).astype(int)

print(f"Test accuracy: {accuracy:.1%}")
print(classification_report(y_test, predictions, target_names=["useless", "useful"]))


Test accuracy: 85.8%
              precision    recall  f1-score   support

     useless       0.80      0.92      0.86       190
      useful       0.92      0.80      0.86       219

    accuracy                           0.86       409
   macro avg       0.86      0.86      0.86       409
weighted avg       0.87      0.86      0.86       409



In [ ]:
#model.save("../exports/linkedin_posts_classifier.keras")

In [ ]:
from pathlib import Path

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

Path( "../exports/linkedin_posts_classifier.tflite").write_bytes(converter.convert())

INFO:tensorflow:Assets written to: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi/assets


INFO:tensorflow:Assets written to: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi/assets


Saved artifact at '/var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 350), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  5841295056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841295440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841296208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841297552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841295632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841296400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5841297936: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1793145043.631600  289812 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793145043.631615  289812 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793145043.632093  289812 reader.cc:83] Reading SavedModel from: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi
I0000 00:00:1793145043.632513  289812 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1793145043.632517  289812 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi
I0000 00:00:1793145043.636255  289812 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1793145043.636802  289812 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1793145043.663650  289812 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/k1/g5m0zxq54mzb_l62jjnwk9th0000gn/T/tmpj_o7e_vi
I0000 00:00:1793145043.671144  289812 lo

9394412

In [ ]:
from google.cloud import storage

project_id = "linkedin-training-512914"
bucket_name = "linkedin-training-512914-models"
# user_id = input("Firebase user ID: ").strip()
# if not user_id:
#     raise ValueError("A Firebase user ID is required")

# prefix = f"models/{user_id}"
# model_object = f"{prefix}/model.tflite"
prefix = f"models/"
model_object = f"model.tflite"

tokenizer_object = f"{prefix}/tokenizer.json"

bucket = storage.Client(project=project_id).bucket(bucket_name)
bucket.blob(model_object).upload_from_filename(
    "../exports/linkedin_posts_classifier.tflite",
    content_type="application/octet-stream"
)
bucket.blob(tokenizer_object).upload_from_string(
    tokenizer.to_json(), content_type="application/json"
)

In [ ]:
from pathlib import Path

Path("../exports/tokenizer.json").write_text(tokenizer.to_json())

586875